# Практика оценки модели: Кредитный скоринг

**Курс:** Метрики и валидация моделей | **Время:** ~60 минут

Вы — ML-инженер в банке. Коллега обучил три модели для задачи кредитного скоринга. Ваша задача — оценить их качество, выявить проблемы и дать рекомендацию.

| Часть | Задача | Баллы |
|-------|--------|-------|
| 1 | Загрузка данных и анализ баланса классов | 15 |
| 2 | Расчёт метрик | 25 |
| 3 | Визуализация и интерпретация | 25 |
| 4 | Диагностика проблем | 25 |
| 5 | Итоговое заключение | 10 |

> Ячейки с `# TODO` — ваш код. Остальные — просто запустите.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
sns.set_style('whitegrid')

---
## Часть 1. Загрузка данных (15 баллов)

In [ ]:
X_raw, y_raw = make_classification(
    n_samples=5000, n_features=15, n_informative=8, n_redundant=3,
    n_clusters_per_class=2, weights=[0.85, 0.15], flip_y=0.02, random_state=42
)

feature_names = [
    'age', 'income', 'credit_history_length', 'num_credit_cards',
    'num_loans', 'loan_amount', 'monthly_payment', 'debt_to_income',
    'employment_years', 'savings_balance', 'checking_balance',
    'num_late_payments', 'credit_utilization', 'num_inquiries', 'collateral_value'
]

df = pd.DataFrame(X_raw, columns=feature_names)
df['target'] = y_raw  # 0 = вернули кредит, 1 = дефолт

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

df.head()

### 1.1 Анализ баланса классов

> **TODO (5 баллов):** Выведите количество и долю каждого класса, постройте bar plot, прокомментируйте результат.

In [ ]:
# TODO

# Вывод: ...

### 1.2 Обучение моделей

In [ ]:
model_a = LogisticRegression(random_state=42, max_iter=1000)
model_a.fit(X_train, y_train)

model_b = RandomForestClassifier(n_estimators=100, random_state=42)
model_b.fit(X, y)

model_c = GradientBoostingClassifier(
    n_estimators=500, max_depth=10, learning_rate=0.5, min_samples_leaf=1, random_state=42
)
model_c.fit(X_train, y_train)

models = {'Модель A (LogReg)': model_a, 'Модель B (RF)': model_b, 'Модель C (GBM)': model_c}
print('Модели обучены')

---
## Часть 2. Расчёт метрик (25 баллов)

### 2.1 Обоснование выбора метрик

> **TODO (5 баллов):** Почему accuracy вводит в заблуждение при дисбалансе классов? Какие метрики важнее всего для кредитного скоринга? Когда важнее Precision, а когда — Recall?

** Ваш ответ:**

...

### 2.2 Вычисление метрик

> **TODO (15 баллов):** Для каждой модели вычислите на тестовой выборке Accuracy, Precision, Recall, F1, ROC-AUC. Выведите итоговую таблицу.

In [ ]:
results = {}

for name, model in models.items():
    y_pred = None  # TODO
    y_prob = None  # TODO
    results[name] = {
        'Accuracy':  None,
        'Precision': None,
        'Recall':    None,
        'F1':        None,
        'ROC-AUC':   None,
    }

# TODO: pd.DataFrame(results).T.round(4)

### 2.3 Classification Report

> **TODO (5 баллов):** Выведите `classification_report` для каждой модели. Сосредоточьтесь на метриках класса `1`.

In [ ]:
for name, model in models.items():
    print(f'\n--- {name} ---')
    # TODO

---
## Часть 3. Визуализация и интерпретация (25 баллов)

### 3.1 Confusion Matrix

> **TODO (10 баллов):** Постройте матрицы ошибок для трёх моделей (1×3 subplots). Какой тип ошибки (FP или FN) критичнее для банка?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, model) in zip(axes, models.items()):
    # TODO
    ax.set_title(name)
plt.tight_layout()
plt.show()

**Анализ:**

...

### 3.2 ROC-кривые

> **TODO (10 баллов):** Постройте ROC-кривые всех трёх моделей на одном графике. Добавьте линию случайного классификатора, AUC в легенду, подпишите оси.

In [ ]:
plt.figure(figsize=(8, 6))
for name, model in models.items():
    # TODO
    pass
# TODO: линия случайного классификатора и оформление
plt.show()

### 3.3 Интерпретация ROC

> **TODO (5 баллов):** Какая модель показывает лучший AUC? Может ли высокий AUC скрывать проблемы? Приведите пример из этого задания.

** Ответ:**

...

---
## Часть 4. Диагностика проблем (25 баллов)

### 4.1 Дисбаланс классов

> **TODO (5 баллов):** Сравните долю предсказанных дефолтов с реальной. Затем обучите `LogisticRegression(class_weight='balanced')` и сравните Recall.

In [ ]:
pred_a = model_a.predict(X_test)
print(f'Доля дефолтов — предсказанная: {pred_a.mean():.3f} | реальная: {y_test.mean():.3f}')

# TODO

### 4.2 Утечка данных (Data Leakage)

> **TODO (10 баллов):** Сравните метрики Модели B на train и test. Объясните, что пошло не так и почему это критично для продакшна.

In [ ]:
# TODO

**Вывод:**

...

### 4.3 Переобучение

> **TODO (10 баллов):** Постройте кривые обучения для Модели C (`learning_curve`, `scoring='roc_auc'`, `cv=5`). По каким признакам на графике видно переобучение?

In [ ]:
# TODO

**Признаки переобучения:**

...

---
## Часть 5. Итоговое заключение (10 баллов)

> **TODO:** Заполните таблицу, сформулируйте рекомендацию и перечислите ограничения.

### Сводная таблица проблем

| Модель | Проблема | Проявление | Критичность |
|--------|----------|------------|--------------|
| A — Logistic Regression | ... | ... | ... |
| B — Random Forest | ... | ... | ... |
| C — Gradient Boosting | ... | ... | ... |

### Рекомендация

...

### Ключевые ограничения

...